In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

crop_path = "../data/processed/crop_yield_clean.csv"
soil_path = "../data/processed/state_soil_data_clean.csv"
weather_path = "../data/processed/state_weather_data_clean.csv"

crop = pd.read_csv(crop_path)
soil = pd.read_csv(soil_path)
weather = pd.read_csv(weather_path)

print("Crop shape:", crop.shape)
print("Soil shape:", soil.shape)
print("Weather shape:", weather.shape)
crop.head(), soil.head(), weather.head()

In [ ]:
# Standardize join keys
for df in (crop, soil, weather):
    df["state"] = df["state"].astype(str).str.strip()

crop["year"] = pd.to_numeric(crop["year"], errors="coerce").astype("Int64")
weather["year"] = pd.to_numeric(weather["year"], errors="coerce").astype("Int64")

print("Unique states in crop not in soil:")
print(sorted(set(crop["state"]) - set(soil["state"])))
print("Unique states in crop not in weather:")
print(sorted(set(crop["state"]) - set(weather["state"])))

In [ ]:
# Merge crop with soil (state-level static features)
crop_soil = crop.merge(soil, on="state", how="left", suffixes=("", "_soil"))
print("After soil merge:", crop_soil.shape)
crop_soil.head()

In [ ]:
# Merge in weather (state-year-level features)
crop_soil_weather = crop_soil.merge(
    weather,
    on=["state", "year"],
    how="left",
    suffixes=("", "_weather"),
)
print("After weather merge:", crop_soil_weather.shape)
crop_soil_weather.head()

In [ ]:
# Basic correlation analysis for yield vs. inputs and climate
feature_cols = [
    "area", "production", "fertilizer", "pesticide", "yield",
    "N", "P", "K", "pH",
    "avg_temp_c", "total_rainfall_mm", "avg_humidity_percent",
]

corr = crop_soil_weather[feature_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
plt.title("Correlation matrix for yield modeling features")
plt.show()

print("Features most correlated with yield:")
display(corr["yield"].sort_values(ascending=False))

In [ ]:
# Example: per-crop feature summary for modeling
per_crop_summary = (
    crop_soil_weather
    .groupby(["crop", "state"], as_index=False)[
        ["year", "yield", "N", "P", "K", "pH", "avg_temp_c", "total_rainfall_mm", "avg_humidity_percent"]
    ]
    .agg({
        "year": ["min", "max", "nunique"],
        "yield": ["mean", "max"],
        "avg_temp_c": "mean",
        "total_rainfall_mm": "mean",
        "avg_humidity_percent": "mean",
        "N": "mean",
        "P": "mean",
        "K": "mean",
        "pH": "mean",
    })
)
per_crop_summary.head()

In [ ]:
# Save ML-ready row-level dataset
import os
os.makedirs("../data/processed", exist_ok=True)

features_path = "../data/processed/crop_yield_ml_features.csv"
crop_soil_weather.to_csv(features_path, index=False)
print("Saved merged ML feature dataset to", features_path)